# BirdCLEF+ 2026 — Exploratory Data Analysis

This notebook inspects metadata and training audio for the Kaggle competition: multilabel species probabilities per 5-second segment on Pantanal soundscapes.

**Adjust `DATA_ROOT` if your data lives elsewhere.**

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from IPython.display import display
from tqdm.auto import tqdm

# --- configuration ---
DATA_ROOT = Path(".").resolve()  # directory containing train.csv, train_audio/, etc.

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("DATA_ROOT =", DATA_ROOT)

In [ ]:
def load_tables(root: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(root / "train.csv")
    tax = pd.read_csv(root / "taxonomy.csv")
    scape_labels = pd.read_csv(root / "train_soundscapes_labels.csv")
    return train, tax, scape_labels


train, taxonomy, soundscape_labels = load_tables(DATA_ROOT)

# train.csv uses column `class_name` (not `class`)
train["primary_label"] = train["primary_label"].astype(str)
taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)

# Expert soundscape annotations are duplicated row-for-row in the released CSV; keep one copy per segment
soundscape_labels_dedup = soundscape_labels.drop_duplicates(
    subset=["filename", "start", "end"], keep="first"
).reset_index(drop=True)

train_audio_root = DATA_ROOT / "train_audio"
audio_species_dirs = sorted([p.name for p in train_audio_root.iterdir() if p.is_dir()]) if train_audio_root.is_dir() else []

print(f"train.csv rows: {len(train):,}")
print(f"taxonomy species: {len(taxonomy)}")
print(f"soundscape label rows (raw): {len(soundscape_labels):,}")
print(f"soundscape segments (unique filename,start,end): {len(soundscape_labels_dedup):,}")
print(f"train_audio subfolders (species with reference clips): {len(audio_species_dirs)}")

## 1. Class distribution — samples per species (sorted bar chart)

Counts are **rows in `train.csv`** (one reference recording per row), keyed by `primary_label`.

In [ ]:
counts = train["primary_label"].value_counts().sort_values(ascending=True)
counts_df = (
    train["primary_label"]
    .value_counts()
    .rename_axis("primary_label")
    .reset_index(name="n_samples")
    .merge(
        taxonomy[["primary_label", "scientific_name", "common_name", "class_name"]],
        on="primary_label",
        how="left",
    )
)

fig, ax = plt.subplots(figsize=(14, max(6, len(counts) * 0.12)))
colors = sns.color_palette("viridis", n_colors=len(counts))
ax.barh(range(len(counts)), counts.values, color=colors)
ax.set_yticks(range(len(counts)))
ax.set_yticklabels(counts.index.astype(str), fontsize=6)
ax.set_xlabel("Number of train.csv rows (reference recordings)")
ax.set_title("Samples per species (sorted ascending)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

display(counts_df.sort_values("n_samples", ascending=False).head(15))
display(counts_df.sort_values("n_samples", ascending=True).head(15))

## 2. Taxonomy breakdown — species count per biological class

In [ ]:
class_counts = taxonomy["class_name"].value_counts()
display(class_counts.to_frame("n_species"))

fig, ax = plt.subplots(figsize=(8, 4))
class_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_ylabel("Number of species in taxonomy")
ax.set_xlabel("Class")
ax.set_title("Taxonomy: species per class (234 total labels)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 3. Soundscape-only species

Species that appear in **`train_soundscapes_labels.csv`** but have **no** reference folder under **`train_audio/`** (no XC/iNat short clips in the downloaded bundle).

In [ ]:
def explode_soundscape_labels(series: pd.Series) -> set[str]:
    out: set[str] = set()
    for cell in series.dropna():
        for part in str(cell).split(";"):
            part = part.strip()
            if part:
                out.add(part)
    return out


labels_in_soundscapes = explode_soundscape_labels(soundscape_labels_dedup["primary_label"])
audio_dir_set = set(audio_species_dirs)
soundscape_only = sorted(labels_in_soundscapes - audio_dir_set)

tax_labels = set(taxonomy["primary_label"].astype(str))
unknown_in_tax = sorted(labels_in_soundscapes - tax_labels)

print(f"Unique label tokens in soundscape annotations: {len(labels_in_soundscapes)}")
print(f"Tokens not in taxonomy.csv: {unknown_in_tax if unknown_in_tax else 'none'}")
print(f"Species (tokens) in soundscapes but NOT in train_audio/: {len(soundscape_only)}")

only_tbl = taxonomy[taxonomy["primary_label"].isin(soundscape_only)].sort_values("class_name")
display(only_tbl)

# Also: taxonomy species with no train_audio at all (should align with soundscape-only union empty ref)
no_audio_in_tax = taxonomy[~taxonomy["primary_label"].isin(audio_dir_set)]["primary_label"].tolist()
print(f"\nTaxonomy rows with no train_audio folder: {len(no_audio_in_tax)}")

## 4. Audio duration statistics (`train_audio/`)

Uses **`soundfile.info`** (header read only; no full waveform decode). First run over ~35k files may take a few minutes.

In [ ]:
ogg_paths = sorted(train_audio_root.rglob("*.ogg"))
records: list[dict] = []
errors: list[tuple[str, str]] = []

for path in tqdm(ogg_paths, desc="Reading .ogg durations"):
    species = path.parent.name
    try:
        info = sf.info(path)
        dur = float(info.duration)
        records.append({"path": str(path), "primary_label": species, "duration_sec": dur, "samplerate": info.samplerate})
    except Exception as e:
        errors.append((str(path), repr(e)))

durations_df = pd.DataFrame.from_records(records)
print(f"Parsed {len(durations_df):,} files; failures: {len(errors)}")
if errors[:5]:
    print("Example errors:", errors[:5])

if len(durations_df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(durations_df["duration_sec"], bins=80, color="coral", edgecolor="white")
    axes[0].set_xlabel("Duration (s)")
    axes[0].set_ylabel("Count (files)")
    axes[0].set_title("All reference recordings: duration distribution")

    merged = durations_df.merge(taxonomy[["primary_label", "class_name"]], on="primary_label", how="left")
    order = merged["class_name"].value_counts().index.tolist()
    sns.boxplot(data=merged, x="class_name", y="duration_sec", order=order, ax=axes[1])
    axes[1].set_xlabel("Class")
    axes[1].set_ylabel("Duration (s)")
    axes[1].set_title("Duration by taxonomic class")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

    per_species = (
        durations_df.groupby("primary_label")["duration_sec"]
        .agg(count="count", mean="mean", std="std", min="min", max="max")
        .reset_index()
        .merge(taxonomy[["primary_label", "scientific_name", "class_name"]], on="primary_label", how="left")
        .sort_values("mean", ascending=False)
    )
    display(per_species.head(12))
    display(per_species.tail(12))

## 5. Rating distribution (`train.csv`)

Xeno-Canto style ratings (higher is typically better). **iNat rows often use `0.0`** as a placeholder.

In [ ]:
r = pd.to_numeric(train["rating"], errors="coerce")
low = (r < 2).sum()
total = r.notna().sum()
print(f"Rows with rating < 2: {low:,} / {total:,} ({100 * low / total:.2f}%)")
print(f"Rows with rating NaN: {r.isna().sum():,}")

fig, ax = plt.subplots(figsize=(10, 4))
r.dropna().hist(bins=50, ax=ax, color="seagreen", edgecolor="white")
ax.axvline(2, color="red", linestyle="--", label="rating = 2")
ax.set_xlabel("Rating")
ax.set_ylabel("Count")
ax.set_title("train.csv rating distribution")
ax.legend()
plt.tight_layout()
plt.show()

by_coll = train.assign(rating=r).groupby("collection")["rating"].agg(
    n="count", frac_below_2=lambda s: (s < 2).mean()
)
display(by_coll)

## 6. Collection split — XC vs iNat overall and per species

In [ ]:
overall = train["collection"].value_counts()
display(overall.to_frame("n_rows"))

ct = (
    train.groupby(["primary_label", "collection"])
    .size()
    .unstack(fill_value=0)
    .rename(columns=lambda c: f"n_{str(c).lower()}")
)
if "n_xc" not in ct.columns:
    ct["n_xc"] = 0
if "n_inat" not in ct.columns:
    ct["n_inat"] = 0
ct = ct.reset_index().merge(
    taxonomy[["primary_label", "scientific_name", "class_name"]], on="primary_label", how="left"
)
ct["n_total"] = ct[[c for c in ct.columns if c.startswith("n_")]].sum(axis=1)
ct["frac_xc"] = ct["n_xc"] / ct["n_total"].replace(0, np.nan)
display(ct.sort_values("n_total", ascending=False).head(20))

fig, ax = plt.subplots(figsize=(10, 4))
overall.plot(kind="bar", ax=ax, color=["#4c72b0", "#dd8452"], edgecolor="black")
ax.set_title("train.csv rows by collection")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Geographic spread — latitude / longitude

In [ ]:
geo = train[["latitude", "longitude", "primary_label", "class_name"]].copy()
geo["latitude"] = pd.to_numeric(geo["latitude"], errors="coerce")
geo["longitude"] = pd.to_numeric(geo["longitude"], errors="coerce")
gvalid = geo.dropna(subset=["latitude", "longitude"])
print(f"Rows with valid lat/lon: {len(gvalid):,} / {len(train):,}")

fig, ax = plt.subplots(figsize=(8, 7))
for cls, sub in gvalid.groupby("class_name"):
    ax.scatter(sub["longitude"], sub["latitude"], s=4, alpha=0.35, label=cls)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("train.csv recording locations (by class_name)")
ax.legend(markerscale=3, frameon=True)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Labeled soundscape segments

Uses **deduplicated** rows (the released CSV repeats each 5 s window twice).

In [ ]:
n_seg = len(soundscape_labels_dedup)
n_files = soundscape_labels_dedup["filename"].nunique()
species_in_scape = explode_soundscape_labels(soundscape_labels_dedup["primary_label"])

print(f"Labeled 5-second segments: {n_seg:,}")
print(f"Unique soundscape files with ≥1 labeled segment: {n_files:,}")
print(f"Unique species / label tokens appearing in annotations: {len(species_in_scape)}")

rows_per_file = soundscape_labels_dedup.groupby("filename").size()
display(rows_per_file.describe().to_frame("segments_per_file"))

## 9. Site metadata from soundscape filenames

Pattern: `BC2026_{Train|Test}_{run}_S{site}_{YYYYMMDD}_{HHMMSS}.ogg` — we extract **`S{site}`** (e.g. `S22`).

In [ ]:
def parse_site_id(filename: str) -> str | None:
    m = re.search(r"_S(\d+)_\d{8}_", filename)
    return f"S{m.group(1)}" if m else None


soundscape_labels_dedup = soundscape_labels_dedup.assign(
    site_id=soundscape_labels_dedup["filename"].map(parse_site_id)
)
missing_site = soundscape_labels_dedup["site_id"].isna().sum()
print(f"Rows without parsed site_id: {missing_site}")

sites_from_labels = soundscape_labels_dedup["site_id"].dropna().unique()
print(f"Unique sites in labeled soundscape filenames: {len(sites_from_labels)}")
display(soundscape_labels_dedup["site_id"].value_counts().head(20))

# Optional: all files present on disk under train_soundscapes/
ts_root = DATA_ROOT / "train_soundscapes"
if ts_root.is_dir():
    on_disk = [p.name for p in ts_root.glob("*.ogg")]
    sites_disk = {parse_site_id(f) for f in on_disk}
    sites_disk.discard(None)
    print(f"\ntrain_soundscapes/ .ogg count: {len(on_disk)}")
    print(f"Unique sites (parsed from on-disk filenames): {len(sites_disk)}")

## 10. Class imbalance severity

Ratio of **most frequent** to **least frequent** species by `train.csv` row counts (species with **at least one** sample).

In [ ]:
c = train["primary_label"].value_counts()
positive = c[c > 0]
imax = positive.idxmax()
imin = positive.idxmin()
ratio = positive.max() / positive.min()
print(f"Species with ≥1 train row: {len(positive)} / {len(taxonomy)} taxonomy entries")
print(f"Most samples: {imax} → {positive.max():,} rows")
print(f"Fewest samples: {imin} → {positive.min():,} rows")
print(f"Imbalance ratio (max/min): {ratio:,.2f}")

zero_in_train = sorted(set(taxonomy["primary_label"]) - set(positive.index.astype(str)))
print(f"\nTaxonomy species with ZERO train.csv rows: {len(zero_in_train)}")
if zero_in_train:
    display(taxonomy[taxonomy["primary_label"].isin(zero_in_train)])